# Training the visual search model, start to finish

This notebook walks through the final model from this repository —
the one that predicts **where the eyes go first** during visual
search — and trains it from scratch **on the actual data**: 114,232
first eye movements from 333 people across 11 published experiments.

**Scope: the first saccade of each trial.** The eyes start at the
screen center, the display appears, and the model predicts where they
go. Later saccades bring in extra machinery (inhibition of return,
changing vantage points) and are out of scope here.

**Input and output.** For each trial the model receives: a picture of
the display (pixels), the goal (the color and shape to look for), and
the history of previous trials. It returns **a probability for every
item on the screen**. One predicted saccade is one random draw.

**The model in one sentence.** Everything — what the display shows,
what you want, what you remember — is written into a single
**priority map**, read through an **attention window** centered on
your eyes, and turned into probabilities by a softmax.

**Before running:** build the dataset once (needs the public data
from https://osf.io/q27ph/):

    python pool_data.py --data_dir ".../Data Files" --out_dir dataset
    python build_contexts.py

Needs `numpy`, `pandas`, `matplotlib`, `torch`. Training takes about
ten minutes on a laptop.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from front_end import render, shape_for, item_positions, IMG
from build_contexts import (opponency_contrast, template_axis,
                            wedge_profiles, NBINS, MAXR)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

radii_np = np.linspace(0.09, MAXR, NBINS)   # distance bins along each ray
pos, _ = item_positions(6)                  # the six item slots (a ring)

## 1. The model, with a diagram

The equation, in words. For each item *i*:

    priority(i) = window(i) x [ goal-weighted evidence(i)
                                + shape(i)
                                + target memory(i) - distractor memory(i) ]

    P(saccade -> i) = softmax over the items

Every source flows into one map; the window (centered on the eyes)
gates all of it; softmax reads the map out. One honest note: from
central fixation all items on the ring are equally far away, so
within this model's scope the window is flat across items - it acts
as a shared gain, doing no selective work. It stays in the equation
because it is theoretically defined (an ego-anchored attention
window), not because first-saccade data constrain its shape.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.2))
boxes = [(0.03, "display\n(pixels)"),
         (0.19, "contrast\nmaps"),
         (0.35, "goal-weighted\nevidence"),
         (0.54, "+ memory\n(2 traces)"),
         (0.73, "x attention\nwindow"),
         (0.885, "softmax")]
for x, label in boxes:
    ax.add_patch(plt.Rectangle((x, 0.35), 0.13, 0.32, fc="#eef2fa",
                               ec="#334488", lw=1.5))
    ax.text(x + 0.065, 0.51, label, ha="center", va="center", fontsize=10)
for x, _ in boxes[:-1]:
    ax.annotate("", xy=(x + 0.17, 0.51), xytext=(x + 0.13, 0.51),
                arrowprops=dict(arrowstyle="->", lw=1.5))
ax.annotate("the goal: 'find the\ngreen diamond'", xy=(0.41, 0.67),
            xytext=(0.35, 0.92), ha="center", fontsize=9,
            arrowprops=dict(arrowstyle="->", color="#118844"))
ax.annotate("previous trials", xy=(0.60, 0.67), xytext=(0.62, 0.92),
            ha="center", fontsize=9,
            arrowprops=dict(arrowstyle="->", color="#aa2222"))
ax.annotate("where the eyes are now", xy=(0.795, 0.35),
            xytext=(0.73, 0.10), ha="center", fontsize=9,
            arrowprops=dict(arrowstyle="->", color="#884411"))
ax.set_xlim(0, 1.08); ax.set_ylim(0, 1); ax.axis("off")
plt.title("one priority map; the window gates everything; softmax reads it out")
plt.show()

## 2. Building it piece by piece

### 2a. From pixels to goal-directed evidence

We make one example display in the model's canonical form (every
subject searched one fixed template all session, so displays are
reconstructed with a CIRCLE target among other shapes, green target
color, red singleton): six items of DIFFERENT shapes - the target is
defined by its shape but never "pops out" as the odd one. We push it through the
perception stage. Three maps come out:

1. the raw **color contrast** map — by convention, red-that-stands-out
   is +, green-that-stands-out is −;
2. the **goal-applied** map: the goal acts *directly on the contrast
   map* — upweight the target color, downweight the distractor color.
   Because the map stores the two colors as + and − of one number,
   both acts are a single multiplication whose *sign the goal
   chooses*: for "find green", flip the map so green is positive and
   red negative (a red-target subject gets no flip). One number does
   both jobs — and in a two-color display that is no restriction,
   because only the green-vs-red *difference* can influence which
   item wins;
3. the **object-presence** map ("something visible is here",
   whatever its color). Shown for completeness: it is nearly equal
   across items, and what is equal everywhere cannot help choose
   BETWEEN items — so the final model drops it.

In [ ]:
TARG, SING = 1, 4
items = [dict(x=pos[j][0], y=pos[j][1],
              color="red" if j == SING else "green",
              shape=shape_for(j + 1, TARG + 1)) for j in range(6)]
img = render(items)
maps = opponency_contrast(img)
u = template_axis("green")           # the goal, as a direction in color space

fig, axes = plt.subplots(1, 4, figsize=(14.5, 3.6))
axes[0].imshow(img, origin="lower")
axes[0].plot(IMG/2, IMG/2, "k+", ms=12)
axes[0].set_title("input: pixels (+ = fixation)", fontsize=10)
v = np.abs(maps["RG"]).max()
axes[1].imshow(maps["RG"], origin="lower", cmap="PiYG", vmin=-v, vmax=v)
axes[1].set_title("color contrast map", fontsize=10)
goal_map = u[0] * maps["RG"] + u[1] * maps["BY"]
v = np.abs(goal_map).max()
axes[2].imshow(goal_map, origin="lower", cmap="PiYG", vmin=-v, vmax=v)
axes[2].set_title("goal-applied: green up (green), red down (pink)", fontsize=10)
axes[3].imshow(maps["P"], origin="lower", cmap="magma")
axes[3].set_title("object-presence map (any contrast)", fontsize=10)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

### A gallery: one example display per study

The dataset pools 11 experiments, and each used its own colors (and
one used four items instead of six). Below, one reconstructed display
per study, built exactly the way the training pipeline builds them —
target diamond among heterogeneous shapes, singleton in the study's
opposite color. (Needs `dataset/saccades_ctx.csv` from the build
steps; Hamblin-Frohman ships no color labels, so it falls back to
green/red.)

In [ ]:
gal = pd.read_csv("dataset/saccades_ctx.csv", low_memory=False,
                  usecols=["study", "setsize", "targCol", "singCol"])
gal = gal[gal.singCol != "none"]

fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for ax, (study, sub) in zip(axes.flat, gal.groupby("study")):
    setsize = int(sub.setsize.mode()[0])
    tc, sc = sub[["targCol", "singCol"]].mode().iloc[0]
    ipos, _ = item_positions(setsize)
    tg, sg = 2, (2 + setsize // 2) % setsize + 1   # target & singleton slots
    items = [dict(x=ipos[j][0], y=ipos[j][1],
                  color=sc if (j + 1) == sg else tc,
                  shape=shape_for(j + 1, tg)) for j in range(setsize)]
    ax.imshow(render(items), origin="lower")
    ax.set_title(f"{study}: {tc}/{sc}, set size {setsize}", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes.flat[gal.study.nunique():]:
    ax.axis("off")
plt.suptitle("reconstructed example displays, one per study", y=0.995)
plt.tight_layout()
plt.show()

### All four stimulus terms, on real displays

Before combining anything, here is each term the model actually uses,
computed from two displays: the real red-singleton case (top) and a
hypothetical blue singleton (bottom). Green = positive, pink =
negative in the signed maps.

- **goal_map** (gain `g_T`): green items positive, red negative — and
  nearly blind to blue, which sits sideways to the green axis;
- **off-goal axis** (gain `g_O`): only the singleton shows, with
  OPPOSITE signs for red vs blue — a signed gain here cannot mean
  "salience";
- **|off-goal|**: the sign stripped — red and blue singletons look
  identical: "odd color, whichever way". Not in the final model
  (within a study it duplicates the distractor-color term); it is the
  channel a bottom-up salience account would need, testable with
  novel singleton colors;
- **presence**: every item, color-blind — and near-identical across
  items, which is why the final model drops it (costing ~nothing).

Note: goal_map and the off-goal axis together ARE the color contrast
map, split in two by the goal's direction — which is why the raw map
is never added again.

In [ ]:
def term_maps(sing_color):
    perp = (-u[1], u[0])
    its = [dict(x=pos[j][0], y=pos[j][1],
                color=sing_color if j == SING else "green",
                shape=shape_for(j + 1, TARG + 1)) for j in range(6)]
    im = render(its)
    m = opponency_contrast(im)
    g = u[0]*m["RG"] + u[1]*m["BY"]
    o = perp[0]*m["RG"] + perp[1]*m["BY"]
    return im, g, o, m["P"]

fig, axes = plt.subplots(2, 5, figsize=(15, 6.2))
titles = ["display", "goal_map (g_T)", "off-goal axis (g_O)",
          "|off-goal| (not used)", "presence (not used)"]
for r, sc in enumerate(["red", "blue"]):
    im, g, o, P = term_maps(sc)
    panels = [im, g, o, np.abs(o), P]
    for c in range(5):
        m = panels[c]
        if c in (1, 2):
            v = np.abs(m).max()
            axes[r, c].imshow(m, origin="lower", cmap="PiYG", vmin=-v, vmax=v)
        elif c == 0:
            axes[r, c].imshow(m, origin="lower")
        else:
            axes[r, c].imshow(m, origin="lower", cmap="magma")
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        if r == 0:
            axes[r, c].set_title(titles[c], fontsize=10)
    axes[r, 0].set_ylabel(f"{sc} singleton", fontsize=9)
plt.tight_layout()
plt.show()

### 2b. Combining the maps with the attention window

Continuing from 2a. The combination is a strict pixel-wise recipe:

    priority(x) = window(x) * ( a * target_color(x)
                                - b * distractor_color(x) )

- `a` enhances everything matching the target color; `b` suppresses
  everything matching the distractor color. The raw contrast map is
  not added again - these two terms ARE the contrast map, projected
  onto the two colors. (Within one two-color study, a and b are
  nearly yoked - only their combination is well identified;
  separating them needs displays with 3+ colors.);
- the drive is **signed** - the red item can go *below* zero,
  actively pushed under the plain-item baseline, not just ignored.
  (Rectified versions were tested and fit worse, and they
  under-reproduce the observed below-baseline suppression: the data
  prefer active suppression over mere relegation.);
- multiply, pixel by pixel, with the window - a field of weight
  centered on the current fixation.

The last step is the only non-pixel-wise one: to choose between
items, the priority map is summed over each item's sector, one
number per item. (The trained pipeline computes this same product
along rays toward each item instead of every pixel - faster, same
math.) The gains and the window shape are placeholders here; training
will learn them.

In [ ]:
a_demo, b_demo = 1.5, 0.5                  # placeholder gains
k_demo, r0_demo = 8.0, 0.55                 # placeholder window shape

perp = (-u[1], u[0])
off_map = perp[0] * maps["RG"] + perp[1] * maps["BY"]
uS = template_axis("red")
cphi = u[0]*uS[0] + u[1]*uS[1]
sphi = perp[0]*uS[0] + perp[1]*uS[1]
distractor_map = cphi * goal_map + sphi * off_map
evidence_map = a_demo * goal_map - b_demo * distractor_map   # signed

cx = cy = IMG / 2
px_per_unit = IMG / 1.5
yy, xx = np.mgrid[0:IMG, 0:IMG]
dist_map = np.hypot(xx - cx, yy - cy) / px_per_unit
window_map = sigmoid(k_demo * (r0_demo - dist_map))

priority_map = window_map * evidence_map

item_priority = np.zeros(6)
ang_map = np.arctan2(yy - cy, xx - cx)
for j in range(6):
    item_ang = np.arctan2(pos[j][1], pos[j][0])
    dd = np.abs((ang_map - item_ang + np.pi) % (2 * np.pi) - np.pi)
    item_priority[j] = priority_map[dd < np.deg2rad(30)].sum()
item_priority /= item_priority.max()

fig, axes = plt.subplots(1, 4, figsize=(14.5, 3.6))
for ax, m, ttl in [(axes[0], evidence_map, "goal-weighted evidence (signed)"),
                   (axes[1], window_map, "x  the window (+ = fixation)"),
                   (axes[2], priority_map, "=  the priority map")]:
    ax.imshow(m, origin="lower", cmap="magma")
    ax.plot(cx, cy, "w+" if m is not evidence_map else "k+", ms=10)
    ax.set_title(ttl, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
cols = ["#888888"] * 6
cols[TARG], cols[SING] = "#118844", "#aa2222"
axes[3].bar(range(1, 7), item_priority, color=cols)
axes[3].set_title("summed per item sector", fontsize=10)
axes[3].set_xlabel("item")
plt.tight_layout()
plt.show()

Read it left to right: the evidence map lights up at every
green item and dips *below* the background at the red one - signed
suppression; the window is a spotlight on the fixation;
their pixel-wise product is the **priority map**; and summing the map
over each item's sector gives one priority per item - the target
highest once its shape evidence is added, the red distractor lowest.
The window's shape (how steep, how far it reaches) and all the gains
are exactly what training will estimate from the real eye
movements.

### 2c. Memory of past trials

Two memories, one line each, updated after every trial: where targets
have been (pulls the eyes, fades fast) and where distractors have
been (pushes the eyes away, fades slowly). Their speeds (`eta`) and
strengths (`beta`) are learned. (No inhibition-of-return term: items
can only be "already inspected" from the second saccade on, outside
this model's scope.)

    h = (1 - eta) * h          # everything fades a little
    h[location] += eta         # today's location gets a boost

## 3. The whole model in five lines

Two evidence profiles per item come from the perception stage:
`D_T` (target-color contrast) and `D_dist` (distractor-color
contrast); `FORM` is the shape evidence, `dist` each item's distance
from fixation, and `hT`, `hD` the memories. Nine numbers to learn.

In [ ]:
def field(D_T, D_dist, FORM, dist, hT, hD, w, radii):
    win_ray = torch.sigmoid(w["k"] * (w["r0"] - radii))         # the window...
    mix = w["a"]*D_T - w["b"]*D_dist                            # signed color drive
    stim = (mix * win_ray).sum(-1)
    win_item = torch.sigmoid(w["k"] * (w["r0"] - dist))         # ...gates everything
    return stim + win_item * (w["g_form"]*FORM
                              + w["beta_T"]*hT + w["beta_D"]*hD)

print("that's the model. softmax(field) is the output.")

## 4. Train on the actual data

The dataset holds every first saccade from 333 people: which item
they looked at first, the display they saw, and the full trial order
(needed to rebuild each person's memories).

We hold out 20% of the *people* — the model never sees them during
training — and fit by maximum likelihood: replay everyone's trials,
score the probability the model gives to each real eye movement, and
nudge all nine numbers by gradient descent. The memory speeds are
learned too, so the memories are rebuilt inside the training loop.

In [ ]:
import data

sacc = pd.read_csv("dataset/saccades_ctx.csv", low_memory=False)
ev = pd.read_csv("dataset/events.csv", low_memory=False)
ctx = np.load("dataset/contexts_v21.npz")
sacc, tt = data.build_tensors(sacc, ev)        # a few minutes: builds tensors

from model import color_angles

cid = torch.tensor(sacc.ctx.values.astype(int))
P = torch.tensor(ctx["P"])[cid]
FORM = torch.tensor(ctx["FORM"])[cid]
cphi, sphi = color_angles(sacc)
D_T = P[..., 0]
D_dist = cphi[:, None, None] * P[..., 0] + sphi[:, None, None] * P[..., 1]
radii = torch.linspace(0.09, 1.1, P.shape[2])
N = len(sacc)
dist = torch.ones(N, 6)
for j in range(1, 7):
    ok = sacc[f"d{j}"].notna().values
    dist[ok, j-1] = torch.tensor(sacc.loc[ok, f"d{j}"].values,
                                 dtype=torch.float32)

n_subj = tt["eT"].shape[0]
rng = np.random.default_rng(0)
test_subj = torch.zeros(n_subj, dtype=torch.bool)
test_subj[rng.choice(n_subj, n_subj // 5, replace=False)] = True
test = test_subj[tt["si"]]
train = ~test
print(f"{N} saccades; {int((~test_subj).sum())} people to train on, "
      f"{int(test_subj.sum())} held out")

In [ ]:
def build_memories(eta_T, eta_D):
    # replay every subject's trials in order; return memory AS OF each trial
    S_, T_, _ = tt["eT"].shape
    hT = torch.zeros(S_, 6); hD = torch.zeros(S_, 6)
    outT, outD = [], []
    for t in range(T_):
        outT.append(hT); outD.append(hD)
        hT = (1 - eta_T) * hT + eta_T * tt["eT"][:, t]
        hD = (1 - eta_D) * hD + eta_D * tt["eD"][:, t]
    return torch.stack(outT, 1), torch.stack(outD, 1)

w = {name: torch.tensor(v, requires_grad=True) for name, v in
     [("a", 0.5), ("b", 0.3), ("g_form", 1.0),
      ("k", 2.0), ("r0", 0.5), ("beta_T", 0.5), ("beta_D", -0.1)]}
raw_eta = {n: torch.tensor(0.0, requires_grad=True) for n in ["T", "D"]}
optimizer = torch.optim.Adam(list(w.values()) + list(raw_eta.values()),
                             lr=0.05)

losses = []
for epoch in range(200):                       # ~10 minutes
    eta_T = torch.sigmoid(raw_eta["T"]); eta_D = torch.sigmoid(raw_eta["D"])
    memT, memD = build_memories(eta_T, eta_D)
    hT = memT[tt["si"], tt["ti"]]; hD = memD[tt["si"], tt["ti"]]
    F = field(D_T, D_dist, FORM, dist, hT, hD, w, radii)
    F = F.masked_fill(~tt["valid"], -1e9)
    logp = torch.log_softmax(F, 1).gather(1, tt["choice"][:, None]).squeeze(1)
    loss = -logp[train].mean()
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    losses.append(loss.item())
    if epoch % 25 == 0:
        print(f"epoch {epoch}: loss {loss.item():.4f}")

plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.xlabel("training step"); plt.ylabel("loss (per saccade)")
plt.title("training on the real eye movements")
plt.show()

In [ ]:
learned = {k: v.item() for k, v in w.items()}
learned["eta_T"] = torch.sigmoid(raw_eta["T"]).item()
learned["eta_D"] = torch.sigmoid(raw_eta["D"]).item()
for name, meaning in [
        ("a", "enhance the target color"),
        ("b", "suppress the distractor color"),
        ("g_form", "goal shape attracts"),
        ("k", "window steepness"), ("r0", "window reach"),
        ("beta_T", "pull toward past target locations"),
        ("beta_D", "push from past distractor locations"),
        ("eta_T", "target memory speed (fast)"),
        ("eta_D", "distractor memory speed (slow)")]:
    print(f"{name:8} = {learned[name]:+7.3f}   {meaning}")

## 5. How good is it? (held-out people only)

We score the model ONLY on the people it never saw. Three
plain-language measures:

- **Probability on the true choice**: how much probability, on
  average, did the model put on the item the person actually looked
  at? (Chance: about 18%.)
- **Top-1 accuracy**: how often was the person's actual choice the
  model's single best guess?
- **Pseudo-R-squared**: 0 means no better than chance, 1 means
  perfect. For models of choices, 0.2-0.4 counts as excellent.

In [ ]:
with torch.no_grad():
    eta_T = torch.sigmoid(raw_eta["T"]); eta_D = torch.sigmoid(raw_eta["D"])
    memT, memD = build_memories(eta_T, eta_D)
    hT = memT[tt["si"], tt["ti"]]; hD = memD[tt["si"], tt["ti"]]
    F = field(D_T, D_dist, FORM, dist, hT, hD, w, radii)
    F = F.masked_fill(~tt["valid"], -1e9)
    prob = torch.softmax(F, 1)
    p_true = prob.gather(1, tt["choice"][:, None]).squeeze(1)

nll = -p_true[test].log().mean()
chance = torch.log(tt["valid"].sum(1).float())[test].mean()
print(f"held-out saccades: {int(test.sum())}")
print(f"mean probability on the true choice: "
      f"{p_true[test].log().mean().exp()*100:.1f}%  "
      f"(chance {torch.exp(-chance)*100:.1f}%)")
print(f"top-1 accuracy: "
      f"{(prob.argmax(1) == tt['choice'])[test].float().mean()*100:.1f}%")
print(f"pseudo-R-squared vs chance: {1 - nll/chance:.3f}")

## 6. Does it behave like people? (held-out people only)

Numbers are one thing; the literature's signature *patterns* are the
real test. Using only the held-out people, we compare observed and
model rates for:

1. **Distractor suppression** — first saccades go to the red
   distractor *less often* than to an average plain item.
2. **Location priming** — saccades to the target roughly double when
   the target repeats its location; saccades to the distractor drop
   when the distractor repeats its location.

Nothing below was fitted to these specific patterns — they come out
of the one trained equation.

In [ ]:
first = torch.tensor((sacc.saccindex == 1).values)  # all True: the scope
sing_present = torch.tensor((sacc.singLoc > 0).values)
rows = torch.arange(N)
isT = torch.zeros(N, 6); isT[rows, torch.tensor(sacc.targLoc.values) - 1] = 1
isS = torch.zeros(N, 6)
sp = torch.tensor(sacc.singLoc.values)
isS[rows[sp > 0], sp[sp > 0] - 1] = 1
chose = torch.zeros(N, 6); chose[rows, tt["choice"]] = 1

m = test & first & sing_present
ot = (chose[m] * isT[m]).sum() / m.sum() * 100
os_ = (chose[m] * isS[m]).sum() / m.sum() * 100
mt = (prob[m] * isT[m]).sum() / m.sum() * 100
ms = (prob[m] * isS[m]).sum() / m.sum() * 100
n_plain = (tt["valid"][m].sum(1).float() - 2).clamp(min=1).mean()
ons = (100 - ot - os_) / n_plain
mns = (100 - mt - ms) / n_plain

x = np.arange(3); width = 0.36
plt.figure(figsize=(6.5, 3.5))
plt.bar(x - width/2, [ot, os_, ons], width, label="people", color="#444444")
plt.bar(x + width/2, [mt, ms, mns], width, label="model", color="#2233aa")
plt.xticks(x, ["target", "red\ndistractor", "plain item\n(average)"])
plt.ylabel("% of first saccades")
plt.title("suppression: the distractor is looked at LESS than a plain item")
plt.legend()
plt.show()

In [ ]:
ev2 = ev.copy()
ev2["block"] = pd.to_numeric(ev2["block"], errors="coerce").fillna(0.0)
ev2["trial"] = pd.to_numeric(ev2["trial"], errors="coerce")
ev2["subj"] = ev2["subj"].astype(str)
ev2 = ev2.sort_values(["study", "subj", "block", "trial"])
ev2["prevT"] = ev2.groupby(["study", "subj"]).targLoc.shift(1)
ev2["prevS"] = ev2.groupby(["study", "subj"]).singLoc.shift(1)
key = ["study", "subj", "block", "trial"]
sacc2 = sacc.merge(ev2[key + ["prevT", "prevS"]], on=key, how="left")

def bar_pair(masks, pick, labels, title, paper):
    obs = [((chose[m_] * pick[m_]).sum() / m_.sum() * 100).item()
           for m_ in masks]
    mod = [((prob[m_] * pick[m_]).sum() / m_.sum() * 100).item()
           for m_ in masks]
    x = np.arange(2)
    plt.bar(x - 0.18, obs, 0.36, label="people", color="#444444")
    plt.bar(x + 0.18, mod, 0.36, label="model", color="#2233aa")
    plt.xticks(x, labels); plt.legend()
    plt.title(f"{title}\n{paper}")

plt.figure(figsize=(11, 3.5))
plt.subplot(1, 2, 1)
tr = test & first & torch.tensor(sacc2.prevT.values == sacc.targLoc.values)
tc = test & first & torch.tensor((sacc2.prevT.values != sacc.targLoc.values)
                                 & ~np.isnan(sacc2.prevT.values))
bar_pair([tr, tc], isT,
         ["target location\nREPEATED", "target location\nchanged"],
         "% of first saccades to the TARGET", "(in the papers: 73 vs 37)")
plt.subplot(1, 2, 2)
sr = test & first & sing_present & torch.tensor(
    (sacc2.prevS.values == sacc.singLoc.values) & (sacc2.prevS.values > 0))
sc = test & first & sing_present & torch.tensor(
    sacc2.prevS.values != sacc.singLoc.values)
bar_pair([sr, sc], isS,
         ["distractor location\nREPEATED", "distractor location\nchanged"],
         "% of first saccades to the DISTRACTOR", "(in the papers: 5 vs 10)")
plt.tight_layout()
plt.show()

## Recap

| The model receives | It returns |
| --- | --- |
| a picture, a goal, the fixation, the trial history | a probability for every item |

We introduced the model and its diagram, built the pieces (perception
-> goal-weighted evidence -> window -> memory), wrote the whole model
in five lines, trained its nine numbers on the real first eye
movements by gradient descent, evaluated it on people it never saw,
and watched it reproduce the literature's signature patterns —
distractor suppression and both location-priming effects — from the
one trained equation.

More: `RESULTS.md` (the full results record) and
`docs/priority_field_visual_search_model.md` (the theory).